# Recipe Extraction Notebook

In [1]:
import json

with open("./parsed_recipes.json", "r") as f:
    parsed_recipes = json.load(f)

In [2]:
# ! TEMPORARY
parsed_recipes = parsed_recipes [:1000]

In [3]:
parsed_recipes [0]

{'directions': ['1. Place the stock, lentils, celery, carrot, thyme, and salt in a medium saucepan and bring to a boil. Reduce heat to low and simmer until the lentils are tender, about 30 minutes, depending on the lentils. (If they begin to dry out, add water as needed.) Remove and discard the thyme. Drain and transfer the mixture to a bowl; let cool.',
  '2. Fold in the tomato, apple, lemon juice, and olive oil. Season with the pepper.',
  '3. To assemble a wrap, place 1 lavash sheet on a clean work surface. Spread some of the lentil mixture on the end nearest you, leaving a 1-inch border. Top with several slices of turkey, then some of the lettuce. Roll up the lavash, slice crosswise, and serve. If using tortillas, spread the lentils in the center, top with the turkey and lettuce, and fold up the bottom, left side, and right side before rolling away from you.'],
 'fat': 7.0,
 'date': '2006-09-01T04:00:00.000Z',
 'categories': ['Sandwich',
  'Bean',
  'Fruit',
  'Tomato',
  'turkey',

In [4]:
import sys
sys.path.append("..")

from food_data_extraction import FoodEmbedding

In [5]:
food_embedding = FoodEmbedding()

c:\Users\micha\Documents\School\University\3rd Year\Final Year Project\Code\recipe_extraction\..\food_data_extraction\FoodEmbedding.py:19: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  self.food_nutrient = pd.read_csv(f"{os.path.join(self.base_dir, 'FoodData_Central_csv_2025-12-18')}/food_nutrient.csv")


In [6]:
food_embedding.load(index_path = "../food_data_extraction/food_embedding.faiss")

In [7]:
NUTRIENT_ID_MAP = {
    "calories": 1008,
    "carbohydrates": 1005,
    "sugar": 1063,
    "protein": 1003,
    "fat": 1004,
    "saturated_fat": 1258,
    "fiber": 1079,
    "sodium": 1093
}

In [8]:
from models import NutritionalInformation

def get_nutritional_information(fdc_id: int | None) -> NutritionalInformation:
    """
    Given FDC food information, this returns the nutritional information for that ingredient based on the FDC data (per 100g)

    :param fdc_id: the FDC food ID for the ingredient
    :type fdc_id: int | None

    :returns: the nutritional information for the ingredient
    :rtype: NutritionalInformation
    """

    # TODO: This method does not check for gluten free, vegan, and dietary tags

    if fdc_id is None:
        return NutritionalInformation()
    
    fdc_nutritional_information = food_embedding.get_nutritional_information(fdc_id)

    nutrients_per_100g = {}

    for nutrient_name, nutrient_id in NUTRIENT_ID_MAP.items(): # TODO should check for other ids, like sugar has 2000 as well
        nutrient = fdc_nutritional_information [fdc_nutritional_information ["nutrient_id"] == nutrient_id]

        if not nutrient.empty:
            nutrients_per_100g [nutrient_name] = nutrient.iloc [0] ["amount"]
        else:
            nutrients_per_100g [nutrient_name] = None

    return NutritionalInformation(
        calories = nutrients_per_100g ["calories"],
        carbohydrates = nutrients_per_100g ["carbohydrates"],
        sugar = nutrients_per_100g ["sugar"],
        protein = nutrients_per_100g ["protein"],
        fat = nutrients_per_100g ["fat"],
        saturated_fat = nutrients_per_100g ["saturated_fat"],
        fiber = nutrients_per_100g ["fiber"],
        sodium = nutrients_per_100g ["sodium"],
        is_gluten_free = None, # TODO
        is_lactose_free = None, # TODO
        is_vegetarian = None, # TODO
        is_vegan = None # TODO
    )

In [9]:
def get_ingredient_weight(fdc_id: int | None, ingredient_weight: float | None) -> float | None:
    """
    Given an ingredient's FDC food result and weight, this returns the weight of that ingredient in grams. This weight is either the weight provided in the recipe, or if that is not available, the default portion size for that ingredient in the FDC data. If neither of those are available, then None is returned
    :param fdc_id: the FDC food ID for the ingredient
    :type fdc_id: int | None
    :param ingredient_weight: the weight of the ingredient in grams, or None if the weight is not known
    :type ingredient_weight: float | None

    :returns: the weight of the ingredient in grams, or None if it cannot be determined
    :rtype: float | None
    """

    if ingredient_weight is not None:
        return ingredient_weight
    
    if fdc_id is None:
        return None

    portion_size = food_embedding.get_portion_size(fdc_id)
    
    if portion_size.empty:
        return 100
    
    return portion_size ["gram_weight"].iloc [0]

In [10]:
from models import DietaryTag

def get_dietary_tags(recipe_categories: list[str]) -> list[DietaryTag]:
    """
    Obtains the dietary tags for a recipe based on its categories

    :param recipe_categories: the categories of the recipe, which may include dietary information such as "Wheat/Gluten-Free", "Dairy-Free", "Vegetarian", and "Vegan"
    :type recipe_categories: list[str]

    :returns: the dietary tags for the recipe
    :rtype: list[DietaryTag]
    """

    dietary_tags: list[DietaryTag] = []

    for category in recipe_categories:
        if category == "Wheat/Gluten-Free":
            dietary_tags.append(DietaryTag.GLUTEN_FREE)
        elif category == "Dairy-Free":
            dietary_tags.append(DietaryTag.LACTOSE_FREE)
        elif category == "Vegetarian":
            dietary_tags.append(DietaryTag.VEGETARIAN)
        elif category == "Vegan":
            dietary_tags.append(DietaryTag.VEGAN)

    return dietary_tags

In [ ]:
from tqdm import tqdm
from models import Ingredient, Recipe

unique_ingredients = dict()

for recipe in tqdm(parsed_recipes [:100], desc = "Extracting Recipes"):
    structured_ingredients = []
    ingredient_amounts = []

    for ingredient_name, ingredient_weight in zip(recipe ["ingredient_names"], recipe ["ingredient_weights"]):
        fdc_id = food_embedding.search(ingredient_name).iloc [0].fdc_id if not food_embedding.search(ingredient_name).empty else None

        if unique_ingredients.get(ingredient_name) is None:
            nutritional_information = get_nutritional_information(fdc_id)

            structured_ingredient = Ingredient(
                name = ingredient_name,
                nutritional_information = nutritional_information
            )

            unique_ingredients [ingredient_name] = structured_ingredient
        else:
            structured_ingredient  = unique_ingredients [ingredient_name]

        ingredient_amount = get_ingredient_weight(fdc_id, ingredient_weight)

        structured_ingredients.append(structured_ingredient)
        ingredient_amounts.append(ingredient_amount)

    ingredient_amount_dict = dict(zip(structured_ingredients, ingredient_amounts))

    total_calories = sum(filter(None, [ingredient.nutritional_information.calories for ingredient in ingredient_amount_dict.keys()]))
    num_serves = total_calories // recipe ["calories"] if recipe ["calories"] else 1

    structured_recipe = Recipe(
        name = recipe ["title"],
        ingredients = ingredient_amount_dict,
        dietary_tags = get_dietary_tags(recipe ["categories"]),
        instructions = recipe ["directions"],
        serves = num_serves
    )
    
    structured_recipe.print()

Extracting Recipes:   0%|          | 0/100 [00:00<?, ?it/s]

For ingredient: low-sodium vegetable stock weight is 946.3529459999997 grams
For ingredient: dried brown lentils weight is 236.58823649999994 grams
For ingredient: dried French green lentils weight is 118.29411824999997 grams
For ingredient: celery weight is 100 grams
For ingredient: carrot weight is 100 grams
For ingredient: fresh thyme weight is 0.8 grams
For ingredient: kosher salt weight is 5.333333333333333 grams
For ingredient: tomato weight is 100 grams
For ingredient: Fuji apple weight is 100 grams
For ingredient: lemon juice weight is 14.0 grams
For ingredient: extra-virgin olive oil weight is 8.333333333333334 grams
For ingredient: black pepper weight is 100 grams
For ingredient: whole-wheat lavash weight is 100 grams
For ingredient: turkey breast weight is 100 grams


Extracting Recipes:   1%|          | 1/100 [00:04<07:12,  4.37s/it]

For ingredient: Bibb lettuce weight is 100 grams
Recipe Title: Lentil, Apple, and Turkey Wrap 
Ingredients: low-sodium vegetable stock, dried brown lentils, dried French green lentils, celery, carrot, fresh thyme, kosher salt, tomato, Fuji apple, lemon juice, extra-virgin olive oil, black pepper, whole-wheat lavash, turkey breast, Bibb lettuce
Dietary Tags: 
Instructions:
1. Place the stock, lentils, celery, carrot, thyme, and salt in a medium saucepan and bring to a boil. Reduce heat to low and simmer until the lentils are tender, about 30 minutes, depending on the lentils. (If they begin to dry out, add water as needed.) Remove and discard the thyme. Drain and transfer the mixture to a bowl; let cool.
2. Fold in the tomato, apple, lemon juice, and olive oil. Season with the pepper.
3. To assemble a wrap, place 1 lavash sheet on a clean work surface. Spread some of the lentil mixture on the end nearest you, leaving a 1-inch border. Top with several slices of turkey, then some of the l

Extracting Recipes:   1%|          | 1/100 [00:05<09:49,  5.95s/it]

For ingredient: whole cloves weight is 100 grams


KeyboardInterrupt: 

In [ ]:
structured_recipe.print()

Recipe Title: Lentil, Apple, and Turkey Wrap 
Ingredients: low-sodium vegetable stock, dried brown lentils, dried French green lentils, celery, carrot, fresh thyme, kosher salt, tomato, Fuji apple, lemon juice, extra-virgin olive oil, black pepper, whole-wheat lavash, turkey breast, Bibb lettuce
Dietary Tags: 
Instructions:
1. Place the stock, lentils, celery, carrot, thyme, and salt in a medium saucepan and bring to a boil. Reduce heat to low and simmer until the lentils are tender, about 30 minutes, depending on the lentils. (If they begin to dry out, add water as needed.) Remove and discard the thyme. Drain and transfer the mixture to a bowl; let cool.
2. Fold in the tomato, apple, lemon juice, and olive oil. Season with the pepper.
3. To assemble a wrap, place 1 lavash sheet on a clean work surface. Spread some of the lentil mixture on the end nearest you, leaving a 1-inch border. Top with several slices of turkey, then some of the lettuce. Roll up the lavash, slice crosswise, and 